In [ ]:
# Cell 1 — Ensure dependencies are available
import importlib
import subprocess
import sys


def _ensure(pkg, import_name=None):
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)


_ensure("timm")
_ensure("librosa")
# onnxruntime is pre-installed on Kaggle CPU notebooks

import librosa
import onnxruntime as ort
import timm

print(f"timm {timm.__version__}  librosa {librosa.__version__}  ort {ort.__version__}")

In [ ]:
# Cell 2 — Imports and config
import glob as _glob
import time
from pathlib import Path

import librosa
import numpy as np
import onnxruntime as ort
import pandas as pd
import soundfile as sf

# ---------------------------------------------------------------------------
# Audio / spectrogram constants — must match train.py exactly
# ---------------------------------------------------------------------------
SR = 32000
FMIN = 20
FMAX = 16000
CLIP_DURATION = 5  # seconds per chunk
CLIP_SAMPLES = SR * CLIP_DURATION

# ---------------------------------------------------------------------------
# Paths — auto-detect by searching for key files
# ---------------------------------------------------------------------------

# Competition data: find taxonomy.csv
_tax_matches = _glob.glob("/kaggle/input/**/taxonomy.csv", recursive=True)
if _tax_matches:
    DATA = Path(_tax_matches[0]).parent
    print(f"Auto-detected DATA: {DATA}")
else:
    DATA = Path("/kaggle/input/birdclef-2026")
    print(f"WARNING: taxonomy.csv not found, using fallback: {DATA}")

# Model dir: find ONNX files
_onnx_matches = _glob.glob("/kaggle/input/**/*.onnx", recursive=True)
if _onnx_matches:
    MODEL_DIR = Path(_onnx_matches[0]).parent
    print(f"Auto-detected MODEL_DIR (ONNX): {MODEL_DIR}")
else:
    MODEL_DIR = Path("/kaggle/input/birdclef2026-soundscape-v7-onnx")
    print(f"WARNING: no .onnx files found, using fallback MODEL_DIR: {MODEL_DIR}")

print(f"DATA:      {DATA}")
print(f"MODEL_DIR: {MODEL_DIR}")

In [ ]:
# Cell 3 — Model definitions and audio utilities (copied from train.py)

import glob as _glob

import torch
import torch.nn as nn
import torch.nn.functional as F

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# ---------------------------------------------------------------------------
# Models
# ---------------------------------------------------------------------------


class BirdModel(nn.Module):
    """EfficientNet (or any timm backbone) with a linear head."""

    def __init__(self, backbone: str, n_classes: int, pretrained: bool = False):
        super().__init__()
        self.backbone = timm.create_model(
            backbone, pretrained=pretrained, num_classes=n_classes, in_chans=3
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)


class BirdModelSED(nn.Module):
    """Naive SED head: attention-weighted pooling over time frames."""

    def __init__(self, backbone: str, n_classes: int, pretrained: bool = False):
        super().__init__()
        self.encoder = timm.create_model(
            backbone, pretrained=pretrained, num_classes=0, global_pool="", in_chans=3
        )
        feat_dim = self.encoder.num_features
        self.fc = nn.Linear(feat_dim, n_classes)
        self.att = nn.Linear(feat_dim, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.encoder(x)
        feat = feat.mean(dim=2)
        feat = feat.permute(0, 2, 1)
        logit = self.fc(feat)
        att = self.att(feat)
        clip = (torch.sigmoid(logit) * torch.sigmoid(att)).sum(1) / (
            torch.sigmoid(att).sum(1) + 1e-7
        )
        return clip


class GEMFreqPool(nn.Module):
    """Generalized Mean Pooling over frequency axis."""

    def __init__(self, p_init: float = 3.0, eps: float = 1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # (B, C, H, W)
        p = self.p.clamp(min=1.0)
        return x.clamp(min=self.eps).pow(p).mean(dim=2).pow(1.0 / p)


class AttentionSEDHead(nn.Module):
    """tanh+softmax attention SED head."""

    def __init__(self, feat_dim: int, num_classes: int, dropout: float = 0.1):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(feat_dim, feat_dim), nn.ReLU(inplace=True), nn.Dropout(dropout)
        )
        self.att_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # (B, C, T)
        x = self.fc(x.permute(0, 2, 1)).permute(0, 2, 1)
        att = F.softmax(torch.tanh(self.att_conv(x)), dim=-1)
        cls = self.cls_conv(x)
        return torch.sigmoid((att * cls).sum(dim=-1))


class BirdModelBaseline(nn.Module):
    """Public baseline replication: GEMFreqPool + AttentionSEDHead."""

    def __init__(
        self,
        backbone: str,
        n_classes: int,
        pretrained: bool = False,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.encoder = timm.create_model(
            backbone, pretrained=pretrained, num_classes=0, global_pool="", in_chans=3
        )
        # Probe actual output dim — num_features is wrong for some backbones (e.g. hgnetv2)
        with torch.no_grad():
            _dummy = torch.zeros(1, 3, 64, 128)
            feat_dim = self.encoder(_dummy).shape[1]
        self.gem_pool = GEMFreqPool(p_init=3.0)
        self.head = AttentionSEDHead(feat_dim, n_classes, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.encoder(x)
        feat = self.gem_pool(feat)
        return self.head(feat)


# ---------------------------------------------------------------------------
# Find BirdSet config (bundled in dataset)
# ---------------------------------------------------------------------------
_cfg_matches = _glob.glob("/kaggle/input/**/*birdset*config*.json", recursive=True)
BIRDSET_CONFIG_PATH = _cfg_matches[0] if _cfg_matches else None
print(f"BirdSet config: {BIRDSET_CONFIG_PATH}")


# ---------------------------------------------------------------------------
# Audio utilities
# ---------------------------------------------------------------------------


def make_melspec(
    y: np.ndarray,
    n_mels: int = 128,
    n_fft: int = 1024,
    hop_length: int = 512,
    fmin: float = FMIN,
    htk: bool = False,
) -> np.ndarray:
    """Compute log-mel spectrogram from a 1-D audio array."""
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=SR,
        n_mels=n_mels,
        hop_length=hop_length,
        n_fft=n_fft,
        fmin=fmin,
        fmax=FMAX,
        htk=htk,
    )
    return librosa.power_to_db(mel, ref=np.max).astype(np.float32)


def spec_to_tensor(spec: np.ndarray) -> torch.Tensor:
    """Z-score normalise + stack to 3-channel tensor."""
    spec = (spec - spec.mean()) / (spec.std() + 1e-6)
    img = np.stack([spec, spec, spec], axis=0)
    t = torch.from_numpy(img)
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t - mean) / std


def spec_to_tensor_minmax(spec: np.ndarray) -> np.ndarray:
    """Per-sample min-max normalise + stack to 3-channel array (baseline model)."""
    mn, mx = spec.min(), spec.max()
    spec = (spec - mn) / (mx - mn + 1e-6)
    return np.stack([spec, spec, spec], axis=0).astype(np.float32)


print("Model classes and audio utilities defined.")

In [ ]:
# Cell 4 — Load species list and ONNX model sessions

import json as _json

taxonomy = pd.read_csv(DATA / "taxonomy.csv")
species = sorted(taxonomy["primary_label"].astype(str).tolist())
n_species = len(species)
print(f"Species count: {n_species}")

# Find ONNX model files
onnx_paths = sorted(MODEL_DIR.glob("*.onnx")) if MODEL_DIR.exists() else []
print(f"ONNX models found: {len(onnx_paths)} in {MODEL_DIR}")

# Configure ONNX Runtime for parallel CPU inference
sess_options = ort.SessionOptions()
sess_options.intra_op_num_threads = 4
sess_options.inter_op_num_threads = 2
sess_options.execution_mode = ort.ExecutionMode.ORT_PARALLEL
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

# models_cfg: list of (session, n_mels, n_fft, hop_length, norm_type, fmin, htk)
models_cfg = []
for onnx_path in onnx_paths:
    meta_path = onnx_path.with_suffix(".json")

    if meta_path.exists():
        meta = _json.loads(meta_path.read_text())
    else:
        meta = {}

    n_mels = meta.get("n_mels", 224)
    n_fft = meta.get("n_fft", 2048)
    hop_length = meta.get("hop_length", 512)
    fmin = meta.get("fmin", 0)
    htk = meta.get("htk", True)
    norm_type = meta.get("norm_type", "minmax")
    val_loss = meta.get("val_loss", float("nan"))
    epoch = meta.get("epoch", "?")

    session = ort.InferenceSession(
        str(onnx_path),
        sess_options=sess_options,
        providers=["CPUExecutionProvider"],
    )
    input_name = session.get_inputs()[0].name
    models_cfg.append(
        (session, input_name, n_mels, n_fft, hop_length, norm_type, fmin, htk)
    )
    print(
        f"  {onnx_path.name} — epoch {epoch}, val_loss {val_loss:.4f}, "
        f"n_mels={n_mels}, hop={hop_length}, fmin={fmin}, htk={htk}"
    )

print(f"\nTotal ONNX models loaded: {len(models_cfg)}")

In [ ]:
# Cell 5 — Inference function (ONNX Runtime + 50% overlap + class-conditional pooling + circular TTA)

TTA_SHIFT_SAMPLES = CLIP_SAMPLES // 4  # 1.25s shift for circular TTA


def _run_windows(y: np.ndarray, models_cfg: list):
    """Run all ONNX models on sliding windows of y. Returns slot_sum, slot_max, slot_counts, n_slots."""
    OVERLAP_STRIDE_S = 2.5
    stride_samples = int(OVERLAP_STRIDE_S * SR)
    positions = list(range(0, len(y) - CLIP_SAMPLES + 1, stride_samples))
    n_slots = len(y) // CLIP_SAMPLES

    if not positions or n_slots == 0:
        return None, None, None, n_slots

    all_model_preds = []
    for (
        session,
        input_name,
        n_mels,
        n_fft,
        hop_length,
        norm_type,
        fmin,
        htk,
    ) in models_cfg:
        # Build batch of spectrograms for all windows
        chunks = []
        for pos in positions:
            chunk = y[pos : pos + CLIP_SAMPLES]
            spec = make_melspec(
                chunk,
                n_mels=n_mels,
                n_fft=n_fft,
                hop_length=hop_length,
                fmin=fmin,
                htk=htk,
            )
            if norm_type == "minmax":
                arr = spec_to_tensor_minmax(spec)  # already numpy
            else:
                arr = spec_to_tensor(spec).numpy()
            chunks.append(arr)

        batch_np = np.stack(chunks)  # (n_windows, 3, n_mels, time)

        # ONNX Runtime inference
        preds = session.run(None, {input_name: batch_np})[0]  # (n_windows, n_species)
        all_model_preds.append(preds)

    window_preds = np.mean(all_model_preds, axis=0)  # (n_windows, n_species)

    slot_sum = np.zeros((n_slots, n_species), dtype=np.float32)
    slot_max = np.full((n_slots, n_species), -np.inf, dtype=np.float32)
    slot_counts = np.zeros(n_slots, dtype=np.int32)

    for w_idx, pos in enumerate(positions):
        w_start = pos / SR
        w_end = w_start + CLIP_DURATION
        for slot in range(n_slots):
            s_start = slot * float(CLIP_DURATION)
            s_end = s_start + float(CLIP_DURATION)
            if w_start < s_end and w_end > s_start:
                slot_sum[slot] += window_preds[w_idx]
                slot_max[slot] = np.maximum(slot_max[slot], window_preds[w_idx])
                slot_counts[slot] += 1

    return slot_sum, slot_max, slot_counts, n_slots


def _aggregate(slot_sum, slot_max, slot_counts, is_event):
    counts = np.maximum(slot_counts, 1)
    slot_mean = slot_sum / counts[:, np.newaxis]
    slot_max = np.where(slot_max == -np.inf, 0.0, slot_max)
    if is_event is not None:
        return np.where(is_event, slot_max, slot_mean)
    return slot_mean


def predict_soundscape(
    path: Path,
    models_cfg: list,
    is_event: np.ndarray | None = None,
    tta: bool = True,
) -> np.ndarray:
    """Sliding-window ONNX inference with 50% overlap + circular TTA.

    Overlap aggregation is class-conditional:
      Event classes (Aves): max pooling — preserves peak call evidence.
      Texture classes (Amphibia/Insecta): mean pooling — averages persistent chorus.

    Returns preds: (n_slots, n_species)
    """
    _t0 = time.perf_counter()
    y, _ = librosa.load(path, sr=SR, mono=True)
    _t_load = time.perf_counter() - _t0
    n_slots = len(y) // CLIP_SAMPLES
    if n_slots == 0 or not models_cfg:
        return np.zeros((max(n_slots, 1), n_species), dtype=np.float32)

    # ── Pass 1: original audio ────────────────────────────────────────────────
    _t1 = time.perf_counter()
    s_sum, s_max, s_cnt, _ = _run_windows(y, models_cfg)
    _t_infer = time.perf_counter() - _t1
    if s_sum is None:
        return np.zeros((n_slots, n_species), dtype=np.float32)
    preds = _aggregate(s_sum, s_max, s_cnt, is_event)
    _t_total = time.perf_counter() - _t0
    print(
        f"    [timing] load={_t_load:.1f}s  infer={_t_infer:.1f}s"
        f"  total={_t_total:.1f}s  ({n_slots} slots)",
        flush=True,
    )

    # ── Pass 2: shifted audio (circular TTA, 1.25s shift) ────────────────────
    if tta:
        y_shift = y[TTA_SHIFT_SAMPLES:]
        n_slots_shift = len(y_shift) // CLIP_SAMPLES
        if n_slots_shift > 0:
            ss_sum, ss_max, ss_cnt, _ = _run_windows(y_shift, models_cfg)
            if ss_sum is not None:
                preds_shift = _aggregate(ss_sum, ss_max, ss_cnt, is_event)
                n_common = min(n_slots, n_slots_shift)
                preds[:n_common] = 0.5 * (preds[:n_common] + preds_shift[:n_common])

    return preds


print(
    "predict_soundscape() defined — ONNX Runtime, 50% overlap, class-conditional pooling, TTA=True."
)

In [ ]:
# Cell 5b — Sanity checks
assert (DATA / "taxonomy.csv").exists(), f"taxonomy.csv not found at {DATA}"
assert (DATA / "sample_submission.csv").exists(), (
    f"sample_submission.csv not found at {DATA}"
)

onnx_paths = sorted(MODEL_DIR.glob("*.onnx")) if MODEL_DIR.exists() else []

tsd = DATA / "test_soundscapes"
soundscape_files = (
    [p for p in tsd.iterdir() if p.suffix in (".ogg", ".wav", ".flac")]
    if tsd.exists()
    else []
)

print(f"DATA:             {DATA}")
print("Taxonomy:         OK")
print(f"ONNX model files: {len(onnx_paths)} .onnx files in {MODEL_DIR}")
print(f"Test soundscapes: {len(soundscape_files)} audio files")

print("\n/kaggle/input (top-level):")
for p in sorted(Path("/kaggle/input").iterdir()):
    print(f"  {'[DIR]' if p.is_dir() else '     '} {p.name}")

In [ ]:
# Cell 6 — Run inference and build submission.csv

sample_sub = pd.read_csv(DATA / "sample_submission.csv")
print(f"Sample submission shape: {sample_sub.shape}")

# ── Taxonomy-based texture/event split ────────────────────────────────────────
TEXTURE_CLASSES = {"Amphibia", "Insecta"}
taxonomy_full = pd.read_csv(DATA / "taxonomy.csv")
label_to_class = dict(
    zip(taxonomy_full["primary_label"].astype(str), taxonomy_full["class_name"])
)
is_texture = np.array(
    [label_to_class.get(sp, "Aves") in TEXTURE_CLASSES for sp in species]
)
is_event = ~is_texture
print(f"Texture species (Amphibia/Insecta): {is_texture.sum()}")
print(f"Event species (Aves/Mammalia/Reptilia): {is_event.sum()}")

ALPHA_TEXTURE = 0.35
ALPHA_EVENT = 0.15
FILE_MAX_WEIGHT = 0.05
TTA = True  # circular TTA enabled — OV speedup makes this feasible

# ── Inference loop ─────────────────────────────────────────────────────────────
test_dir = DATA / "test_soundscapes"
soundscapes = (
    sorted([p for p in test_dir.iterdir() if p.suffix in (".ogg", ".wav", ".flac")])
    if test_dir.exists()
    else []
)
print(f"\nFound {len(soundscapes)} test soundscapes in {test_dir}")
print(f"TTA: {TTA} | Models: {len(models_cfg)}")

rows = {}
_t_loop_start = time.perf_counter()
for _sc_idx, sc_path in enumerate(soundscapes):
    _t_file_start = time.perf_counter()
    print(f"  [{_sc_idx + 1}/{len(soundscapes)}] {sc_path.name} ...", flush=True)
    preds = predict_soundscape(sc_path, models_cfg, is_event=is_event, tta=TTA)

    # ── Persistence penalty ────────────────────────────────────────────────
    if len(preds) > 2:
        nbr_mean = np.empty_like(preds)
        nbr_mean[0] = preds[1]
        nbr_mean[-1] = preds[-2]
        nbr_mean[1:-1] = 0.5 * (preds[:-2] + preds[2:])
        spike = np.maximum(preds - nbr_mean, 0.0)
        excess = np.maximum(spike - 0.30, 0.0)
        preds[:, is_texture] -= 0.40 * excess[:, is_texture]
        preds = np.clip(preds, 0.0, 1.0)

    # ── Temporal smoothing ────────────────────────────────────────────────
    if len(preds) > 1:
        smooth = preds.copy()
        smooth[1:-1][:, is_texture] = (
            (1 - 2 * ALPHA_TEXTURE) * preds[1:-1][:, is_texture]
            + ALPHA_TEXTURE * preds[:-2][:, is_texture]
            + ALPHA_TEXTURE * preds[2:][:, is_texture]
        )
        smooth[0][is_texture] = (1 - ALPHA_TEXTURE) * preds[0][
            is_texture
        ] + ALPHA_TEXTURE * preds[1][is_texture]
        smooth[-1][is_texture] = (1 - ALPHA_TEXTURE) * preds[-1][
            is_texture
        ] + ALPHA_TEXTURE * preds[-2][is_texture]
        local_max = preds.copy()
        local_max[1:-1] = np.maximum(preds[1:-1], np.maximum(preds[:-2], preds[2:]))
        local_max[0] = np.maximum(preds[0], preds[1])
        local_max[-1] = np.maximum(preds[-1], preds[-2])
        smooth[:, is_event] = (1 - ALPHA_EVENT) * preds[
            :, is_event
        ] + ALPHA_EVENT * local_max[:, is_event]
        preds = smooth

    # ── File-max prior ─────────────────────────────────────────────────────
    file_max = preds.max(axis=0, keepdims=True)
    preds = preds + FILE_MAX_WEIGHT * file_max

    for i, chunk_preds in enumerate(preds):
        end_sec = (i + 1) * 5
        rows[f"{sc_path.stem}_{end_sec}"] = chunk_preds

    _t_file = time.perf_counter() - _t_file_start
    _elapsed = time.perf_counter() - _t_loop_start
    _avg_s = _elapsed / (_sc_idx + 1)
    _eta_s = _avg_s * (len(soundscapes) - _sc_idx - 1)
    print(
        f"    -> {len(preds)} chunks | file={_t_file:.1f}s | "
        f"elapsed={_elapsed / 60:.1f}min | ETA={_eta_s / 60:.1f}min",
        flush=True,
    )

print(f"\nTotal rows from inference: {len(rows)}")

# ── Build submission ──────────────────────────────────────────────────────────
expected_species = [c for c in sample_sub.columns if c != "row_id"]
out_rows = []
for _, r in sample_sub.iterrows():
    rid = r["row_id"]
    if rid in rows:
        preds_row = rows[rid]
        row = {"row_id": rid}
        for j, sp in enumerate(species):
            row[sp] = float(preds_row[j]) if j < len(preds_row) else 0.0
    else:
        row = {"row_id": rid, **{sp: 0.0 for sp in species}}
    out_rows.append(row)

sub = pd.DataFrame(out_rows)
for c in expected_species:
    if c not in sub.columns:
        sub[c] = 0.0
sub = sub[["row_id"] + expected_species]

out_path = Path("submission.csv")
sub.to_csv(out_path, index=False)
print(f"\nSubmission saved: {out_path}  shape={sub.shape}")
print(sub.head(3))